# Chapter 06: Kernel Fusion & torch.compile

**Goal:** Understand why unfused elementwise ops waste memory bandwidth and how kernel fusion fixes it.

In Chapter 2, we saw elementwise ops (RMSNorm, SiLU) sit at ~1-2 FLOPS/byte — **100x below the ridge point**.
Each PyTorch op launches a separate CUDA kernel, and each kernel does a full HBM roundtrip.
Fusion combines multiple ops into one kernel, eliminating intermediate reads/writes.

**Model:** LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers)

**Approach:** Fill in every `# YOUR CODE` section yourself — that's the learning.

**Hardware:** A100 80GB SXM4
- Measured HBM bandwidth: 1774 GB/s
- FP16 TFLOPS: 237
- Ridge point: ~134 FLOPS/byte

## 1. The Fusion Problem

In Chapter 2, we profiled a transformer block and found elementwise ops at ~1-2 FLOPS/byte — 100x below
the A100's ridge point of ~134 FLOPS/byte. Why so low?

**Each PyTorch op launches a separate CUDA kernel with its own HBM roundtrip.**

Consider this chain: `RMSNorm -> dropout -> residual add`

```
Kernel 1 (RMSNorm):   Read x from HBM -> compute -> Write result to HBM
Kernel 2 (Dropout):   Read result from HBM -> compute -> Write result to HBM
Kernel 3 (Add):       Read result + residual from HBM -> compute -> Write result to HBM
```

That's **3 reads + 3 writes** when we could do **1 read + 1 write** if all three ops lived in one kernel.
The compute is trivial (a few FLOPS per element) — it's the HBM traffic that dominates.

**Fusion = fewer HBM roundtrips = same compute, fewer bytes = higher arithmetic intensity.**

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda')
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoConfig

model_id = "meta-llama/Llama-3.2-1B"
config = AutoConfig.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
model.eval()

block = model.model.layers[0]

d_model = config.hidden_size       # 2048
d_ff = config.intermediate_size    # 8192
n_heads = config.num_attention_heads   # 32
n_kv_heads = config.num_key_value_heads  # 8
d_head = d_model // n_heads        # 64

print(f"d_model={d_model}, d_ff={d_ff}, {n_heads} Q heads, {n_kv_heads} KV heads")

## 2. Manual Fusion Baseline

Let's measure the cost of unfused vs fused elementwise chains.

**Unfused chain:** RMSNorm -> SiLU -> elementwise multiply (3 separate kernels)

**Fused version:** a single function that does all three ops in one pass.

The GPU has to load the same data 3x in the unfused version but only 1x in the fused version.

In [ ]:
# Helper: precise CUDA timing
def cuda_timer(fn, warmup=10, repeats=100):
    """Time a function on CUDA with proper synchronization."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats  # ms

x = torch.randn(512, 2048, dtype=torch.float16, device=device)
weight = torch.randn(2048, dtype=torch.float16, device=device)
gate = torch.randn(512, 2048, dtype=torch.float16, device=device)

# --- YOUR CODE: time unfused chain (3 separate ops) ---
# Hint: RMSNorm manually = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight
# Hint: SiLU = torch.nn.functional.silu(normed)
# Hint: gate multiply = silu_out * gate

def unfused_chain(x, weight, gate):
    # YOUR CODE
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight  # Hint: manual RMSNorm
    activated = torch.nn.functional.silu(normed)  # Hint: SiLU activation
    return activated * gate  # Hint: elementwise gate multiply

# --- YOUR CODE: time each op separately ---
# Hint: time_norm = cuda_timer(lambda: x * torch.rsqrt(...) * weight)
time_norm = cuda_timer(lambda: x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight)
time_silu = cuda_timer(lambda: torch.nn.functional.silu(x))
time_mul = cuda_timer(lambda: x * gate)
time_unfused = cuda_timer(lambda: unfused_chain(x, weight, gate))

print(f"Individual op times:")
print(f"  RMSNorm:   {time_norm:.4f} ms")
print(f"  SiLU:      {time_silu:.4f} ms")
print(f"  Multiply:  {time_mul:.4f} ms")
print(f"  Sum:       {time_norm + time_silu + time_mul:.4f} ms")
print(f"  Unfused:   {time_unfused:.4f} ms")

# --- YOUR CODE: fuse into a single function ---
# Hint: do all 3 operations in one function call
def fused_chain(x, weight, gate):
    # YOUR CODE: combine all three ops into one function
    # Hint: same math, but torch may still launch separate kernels
    return torch.nn.functional.silu(
        x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight
    ) * gate

time_fused = cuda_timer(lambda: fused_chain(x, weight, gate))
print(f"\nFused call:  {time_fused:.4f} ms")
print(f"Speedup:     {time_unfused / time_fused:.2f}x")
print(f"\nNote: manual fusion in Python doesn't truly fuse CUDA kernels.")
print(f"We need torch.compile or Triton for real kernel fusion.")

## 3. torch.compile Basics

`torch.compile(model)` uses **TorchInductor** to automatically fuse elementwise ops into a single kernel.
It traces the Python code into an FX graph and generates optimized Triton kernels.

This is the easiest way to get fusion — one line of code.

In [ ]:
# Benchmark: eager vs compiled transformer block
seq_len = 512
x = torch.randn(1, seq_len, d_model, dtype=torch.float16, device=device)

# Pre-compute RoPE embeddings
position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
rotary_emb = model.model.layers[0].self_attn.rotary_emb
position_embeddings = rotary_emb(x, position_ids)

# Eager mode baseline
def run_eager():
    with torch.no_grad():
        return block(x, position_embeddings=position_embeddings, use_cache=False)

time_eager = cuda_timer(run_eager)
print(f"Eager mode: {time_eager:.3f} ms")

# --- YOUR CODE: compile the block and benchmark ---
# Hint: compiled_block = torch.compile(block)
compiled_block = torch.compile(block)  # YOUR CODE

def run_compiled():
    with torch.no_grad():
        return compiled_block(x, position_embeddings=position_embeddings, use_cache=False)

# First call triggers compilation (slow)
print("Compiling (first call)...")
_ = run_compiled()
torch.cuda.synchronize()

time_compiled = cuda_timer(run_compiled)
print(f"Compiled mode: {time_compiled:.3f} ms")
print(f"Speedup: {time_eager / time_compiled:.2f}x")

In [ ]:
# --- YOUR CODE: use torch._dynamo.explain to see what gets fused ---
# Hint: explanation = torch._dynamo.explain(block)(x, position_embeddings=position_embeddings, use_cache=False)

explanation = torch._dynamo.explain(block)(  # YOUR CODE
    x, position_embeddings=position_embeddings, use_cache=False
)

print(f"Number of graphs: {explanation.graph_count}")
print(f"Number of graph breaks: {explanation.break_count}")
print(f"\nBreak reasons:")
for reason in explanation.break_reasons:
    print(f"  - {reason}")

## 4. Understanding TorchInductor

How `torch.compile` works under the hood:

```
Python code -> TorchDynamo -> FX Graph -> TorchInductor -> Triton kernels -> CUDA
```

1. **TorchDynamo** traces Python into an FX intermediate representation
2. **TorchInductor** analyzes the FX graph and fuses compatible ops
3. Inductor generates **Triton** code (GPU kernel language by OpenAI)
4. Triton compiles to PTX/CUBIN that runs on the GPU

We can inspect the generated code to see which ops got fused.

In [ ]:
import os
import torch._inductor.config as inductor_config

# --- YOUR CODE: enable debug output to see generated Triton code ---
# Hint: set TORCH_COMPILE_DEBUG=1 or use inductor_config
inductor_config.debug = True  # YOUR CODE: enable Inductor debug output

# Simple example to inspect: elementwise chain
def elementwise_chain(x, weight, gate):
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight
    activated = torch.nn.functional.silu(normed)
    return activated * gate

# YOUR CODE: compile and run to trigger code generation
compiled_chain = torch.compile(elementwise_chain)  # YOUR CODE

test_x = torch.randn(512, 2048, dtype=torch.float16, device=device)
test_w = torch.randn(2048, dtype=torch.float16, device=device)
test_g = torch.randn(512, 2048, dtype=torch.float16, device=device)

# Trigger compilation
_ = compiled_chain(test_x, test_w, test_g)
torch.cuda.synchronize()

# Compare unfused vs compiled
time_unfused_elem = cuda_timer(lambda: elementwise_chain(test_x, test_w, test_g))
time_compiled_elem = cuda_timer(lambda: compiled_chain(test_x, test_w, test_g))

print(f"Elementwise chain:")
print(f"  Unfused:   {time_unfused_elem:.4f} ms")
print(f"  Compiled:  {time_compiled_elem:.4f} ms")
print(f"  Speedup:   {time_unfused_elem / time_compiled_elem:.2f}x")
print(f"\nInductor fuses all elementwise ops into a single Triton kernel.")
print(f"Check torch_compile_debug/ directory for generated Triton code.")

inductor_config.debug = False  # Reset

## 5. CUDA Graphs

Even after kernel fusion, there's still **Python overhead** between kernel launches.
Each `torch.matmul()` call goes: Python -> PyTorch dispatcher -> CUDA driver -> GPU.

**CUDA Graphs** solve this by:
1. **Capture:** Record a sequence of GPU operations into a graph
2. **Replay:** Execute the entire graph with a single CPU call

This eliminates CPU-side overhead entirely — the GPU just replays the captured sequence.

Most impactful when you have many small kernels (like in decode).

In [ ]:
# --- YOUR CODE: implement CUDA graph capture and replay ---

# Static inputs (CUDA graphs require fixed-size tensors)
static_x = torch.randn(1, seq_len, d_model, dtype=torch.float16, device=device)

# Warmup (required before capture)
with torch.no_grad():
    for _ in range(3):
        _ = block(static_x, position_embeddings=position_embeddings, use_cache=False)

# --- YOUR CODE: capture CUDA graph ---
# Hint: g = torch.cuda.CUDAGraph()
# Hint: with torch.cuda.graph(g): ...
g = torch.cuda.CUDAGraph()  # YOUR CODE

with torch.no_grad():
    with torch.cuda.graph(g):  # YOUR CODE: capture the forward pass
        static_output = block(
            static_x,
            position_embeddings=position_embeddings,
            use_cache=False
        )

# --- YOUR CODE: benchmark graph replay vs normal execution ---
# Hint: g.replay()
def run_normal():
    with torch.no_grad():
        return block(static_x, position_embeddings=position_embeddings, use_cache=False)

def run_graph():
    g.replay()  # YOUR CODE: replay the captured graph
    return static_output

time_normal = cuda_timer(run_normal)
time_graph = cuda_timer(run_graph)

print(f"Normal execution: {time_normal:.3f} ms")
print(f"CUDA graph replay: {time_graph:.3f} ms")
print(f"Speedup: {time_normal / time_graph:.2f}x")
print(f"\nNote: to change inputs, copy into static_x (same memory address).")
print(f"CUDA graphs replay the exact same kernel sequence with the exact same pointers.")

## 6. Fusion Patterns That Matter

torch.compile fuses specific patterns well:

| Pattern | Example | Why it helps |
|---------|---------|-------------|
| Elementwise chains | norm -> activation -> dropout | Eliminates intermediate HBM writes |
| Norm + activation | RMSNorm + SiLU | Two memory-bound ops become one |
| Residual connections | output + residual | Avoids separate add kernel |

It does **not** fuse across matmul boundaries (matmuls use cuBLAS, not Triton).

In [ ]:
# --- YOUR CODE: create examples of each fusion pattern and verify ---

x = torch.randn(512, 2048, dtype=torch.float16, device=device)
residual = torch.randn_like(x)
w_norm = torch.randn(2048, dtype=torch.float16, device=device)

# Pattern (a): Elementwise chain — norm + activation + dropout
def pattern_a_eager(x, w_norm):
    # YOUR CODE
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * w_norm  # Hint: RMSNorm
    activated = torch.nn.functional.silu(normed)  # Hint: SiLU
    dropped = torch.nn.functional.dropout(activated, p=0.1, training=True)  # Hint: dropout
    return dropped

# Pattern (b): Norm + residual add
def pattern_b_eager(x, w_norm, residual):
    # YOUR CODE
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * w_norm  # Hint: RMSNorm
    return normed + residual  # Hint: residual add

# Pattern (c): Full residual block
def pattern_c_eager(x, w_norm, residual):
    # YOUR CODE
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * w_norm  # Hint: RMSNorm
    activated = torch.nn.functional.silu(normed)  # Hint: SiLU
    return activated + residual  # Hint: residual add

# --- YOUR CODE: compile each pattern ---
pattern_a_compiled = torch.compile(pattern_a_eager)  # YOUR CODE
pattern_b_compiled = torch.compile(pattern_b_eager)  # YOUR CODE
pattern_c_compiled = torch.compile(pattern_c_eager)  # YOUR CODE

# Warmup compiled versions
_ = pattern_a_compiled(x, w_norm)
_ = pattern_b_compiled(x, w_norm, residual)
_ = pattern_c_compiled(x, w_norm, residual)
torch.cuda.synchronize()

# Benchmark all patterns
results = []
for name, eager_fn, compiled_fn, args in [
    ("(a) Norm+SiLU+Dropout", pattern_a_eager, pattern_a_compiled, (x, w_norm)),
    ("(b) Norm+Residual", pattern_b_eager, pattern_b_compiled, (x, w_norm, residual)),
    ("(c) Norm+SiLU+Residual", pattern_c_eager, pattern_c_compiled, (x, w_norm, residual)),
]:
    t_eager = cuda_timer(lambda fn=eager_fn, a=args: fn(*a))
    t_compiled = cuda_timer(lambda fn=compiled_fn, a=args: fn(*a))
    results.append((name, t_eager, t_compiled))
    print(f"{name}:")
    print(f"  Eager:    {t_eager:.4f} ms")
    print(f"  Compiled: {t_compiled:.4f} ms")
    print(f"  Speedup:  {t_eager / t_compiled:.2f}x")
    print()

## 7. When torch.compile Helps Most

torch.compile has a fixed compilation overhead (can be 10-60s).
After that, each call is faster. The tradeoff:

- **Small batch sizes / few calls:** compile overhead dominates -> not worth it
- **Large batch sizes / many calls:** runtime savings accumulate -> big win

Let's benchmark across batch sizes to see the crossover.

In [ ]:
# --- YOUR CODE: benchmark compiled model across batch sizes ---
# Hint: try batch_sizes = [1, 8, 32, 128]

batch_sizes = [1, 8, 32, 128]  # YOUR CODE
results_batch = []

for bs in batch_sizes:
    x_bs = torch.randn(bs, seq_len, d_model, dtype=torch.float16, device=device)
    pos_ids = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, -1)
    pos_emb = rotary_emb(x_bs, pos_ids)

    # Eager
    def run_eager_bs(x_in=x_bs, pe=pos_emb):
        with torch.no_grad():
            return block(x_in, position_embeddings=pe, use_cache=False)

    t_eager = cuda_timer(run_eager_bs)

    # Compiled (reuse same compiled_block from earlier)
    def run_compiled_bs(x_in=x_bs, pe=pos_emb):
        with torch.no_grad():
            return compiled_block(x_in, position_embeddings=pe, use_cache=False)

    # Trigger recompilation for new shape
    try:
        _ = run_compiled_bs()
        torch.cuda.synchronize()
        t_compiled = cuda_timer(run_compiled_bs)
    except Exception as e:
        print(f"  Batch {bs}: compilation failed ({e}), using eager time")
        t_compiled = t_eager

    results_batch.append({
        'batch_size': bs,
        'eager_ms': t_eager,
        'compiled_ms': t_compiled,
        'speedup': t_eager / t_compiled
    })
    print(f"Batch {bs:>3d}: eager={t_eager:.3f} ms, compiled={t_compiled:.3f} ms, speedup={t_eager/t_compiled:.2f}x")

# --- YOUR CODE: calculate compile-time vs runtime tradeoff ---
# Hint: if compile takes ~30s and saves 0.1ms/call, need 300K calls to break even
print(f"\nCompile-time tradeoff:")
compile_overhead_s = 30  # typical compile time in seconds
for r in results_batch:
    saving_per_call_ms = r['eager_ms'] - r['compiled_ms']
    if saving_per_call_ms > 0:
        calls_to_break_even = int(compile_overhead_s * 1000 / saving_per_call_ms)  # YOUR CODE
        print(f"  Batch {r['batch_size']:>3d}: saves {saving_per_call_ms:.3f} ms/call -> break even after {calls_to_break_even:,} calls")
    else:
        print(f"  Batch {r['batch_size']:>3d}: no speedup")

## 8. Limitations and Gotchas

torch.compile doesn't work for everything. Key issues:

1. **Graph breaks:** Dynamic control flow, data-dependent shapes, unsupported Python features
2. **Dynamic shapes:** Different input shapes trigger recompilation
3. **Custom ops:** Some custom CUDA ops don't trace through Dynamo

A graph break splits the model into multiple subgraphs, reducing fusion opportunities.

In [ ]:
# --- YOUR CODE: demonstrate a graph break and show how to fix it ---

# Example: data-dependent control flow causes a graph break
def bad_function(x):
    """This causes a graph break: Python if on tensor value."""
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)
    if x.mean() > 0:  # YOUR CODE: data-dependent branch causes graph break
        return torch.nn.functional.silu(normed)
    else:
        return torch.nn.functional.gelu(normed)

# Explain the graph break
test_x = torch.randn(512, 2048, dtype=torch.float16, device=device)
explanation_bad = torch._dynamo.explain(bad_function)(test_x)
print(f"BAD function (data-dependent if):")
print(f"  Graph count: {explanation_bad.graph_count}")
print(f"  Break count: {explanation_bad.break_count}")

# --- YOUR CODE: fix the graph break ---
# Hint: use torch.where() instead of Python if/else
def fixed_function(x):
    """Fixed: use torch.where instead of Python if."""
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)
    silu_out = torch.nn.functional.silu(normed)
    gelu_out = torch.nn.functional.gelu(normed)
    return torch.where(x.mean() > 0, silu_out, gelu_out)  # YOUR CODE: torch.where avoids graph break

explanation_fixed = torch._dynamo.explain(fixed_function)(test_x)
print(f"\nFIXED function (torch.where):")
print(f"  Graph count: {explanation_fixed.graph_count}")
print(f"  Break count: {explanation_fixed.break_count}")

# Benchmark
compiled_bad = torch.compile(bad_function)
compiled_fixed = torch.compile(fixed_function)
_ = compiled_bad(test_x)  # trigger compilation
_ = compiled_fixed(test_x)
torch.cuda.synchronize()

t_bad = cuda_timer(lambda: compiled_bad(test_x))
t_fixed = cuda_timer(lambda: compiled_fixed(test_x))
print(f"\nCompiled bad:   {t_bad:.4f} ms")
print(f"Compiled fixed: {t_fixed:.4f} ms")
print(f"Fixing graph break speedup: {t_bad / t_fixed:.2f}x")

## 9. Roofline Impact

How does fusion change the roofline picture?

**Unfused:** Each op reads/writes HBM separately -> many bytes -> low intensity

**Fused:** One kernel does all ops -> one read + one write -> same FLOPs, fewer bytes -> **higher intensity**

Fusion moves ops **rightward** on the roofline (toward the ridge point).

In [ ]:
# --- YOUR CODE: plot unfused vs fused elementwise ops on roofline ---

peak_bw = 1774   # GB/s
peak_tfl = 237    # TFLOPS
ridge_point = peak_tfl * 1000 / peak_bw  # ~134 FLOPS/byte

S, D = 512, 2048
elements = S * D

# RMSNorm -> SiLU -> gate multiply chain
total_flops = (5 + 4 + 1) * elements  # ~10 FLOPS per element (norm: 5, silu: 4, mul: 1)

# --- YOUR CODE: calculate bytes for unfused (3 separate kernels) ---
# Hint: each kernel reads input from HBM and writes output to HBM
# RMSNorm: read x + write normed = 2 * elements * 2 bytes, plus read weight
# SiLU:    read normed + write activated = 2 * elements * 2 bytes
# Mul:     read activated + read gate + write output = 3 * elements * 2 bytes
bytes_unfused = 2 * (2 * elements + 2 * elements + 3 * elements + D)  # YOUR CODE: ~7 element-sized transfers
intensity_unfused = total_flops / bytes_unfused  # YOUR CODE

# --- YOUR CODE: calculate bytes for fused (1 kernel) ---
# Hint: read x + read weight + read gate + write output = ~3 element transfers + weight
bytes_fused = 2 * (elements + D + elements + elements)  # YOUR CODE: read x, weight, gate; write output
intensity_fused = total_flops / bytes_fused  # YOUR CODE

print(f"Chain: RMSNorm -> SiLU -> gate multiply on ({S}, {D})")
print(f"Total FLOPs: {total_flops:,}")
print(f"")
print(f"Unfused (3 kernels):")
print(f"  Bytes: {bytes_unfused:,} ({bytes_unfused / 1e6:.2f} MB)")
print(f"  Intensity: {intensity_unfused:.2f} FLOPS/byte")
print(f"")
print(f"Fused (1 kernel):")
print(f"  Bytes: {bytes_fused:,} ({bytes_fused / 1e6:.2f} MB)")
print(f"  Intensity: {intensity_fused:.2f} FLOPS/byte")
print(f"")
print(f"Byte reduction: {bytes_unfused / bytes_fused:.1f}x fewer bytes")
print(f"Intensity increase: {intensity_fused / intensity_unfused:.1f}x")
print(f"Ridge point: {ridge_point:.0f} FLOPS/byte")
print(f"Both are still memory-bound (intensity << {ridge_point:.0f}), but fused is closer to ridge.")

In [ ]:
# Plot unfused vs fused on roofline
intensity_range = np.logspace(-1, 3, 1000)
memory_bound = peak_bw / 1000 * intensity_range  # TFLOPS
roofline = np.minimum(memory_bound, peak_tfl)

fig, ax = plt.subplots(figsize=(12, 8))
ax.loglog(intensity_range, roofline * 1000, 'k-', linewidth=2.5, label='A100 Roofline', zorder=10)

# Shade regions
mem_mask = intensity_range < ridge_point
ax.fill_between(intensity_range[mem_mask], roofline[mem_mask] * 1000,
                alpha=0.15, color='blue', label='Memory-bound region')
ax.fill_between(intensity_range[~mem_mask], roofline[~mem_mask] * 1000,
                alpha=0.15, color='red', label='Compute-bound region')
ax.axvline(ridge_point, color='red', linestyle='--', linewidth=1, alpha=0.5)

# Plot unfused and fused points
achieved_unfused = peak_bw * intensity_unfused * 0.5  # memory-bound estimate
achieved_fused = peak_bw * intensity_fused * 0.5

ax.loglog(intensity_unfused, achieved_unfused, 'rv', markersize=14, label='Unfused (3 kernels)', zorder=5)
ax.loglog(intensity_fused, achieved_fused, 'g^', markersize=14, label='Fused (1 kernel)', zorder=5)

# Arrow showing the improvement
ax.annotate('', xy=(intensity_fused, achieved_fused),
            xytext=(intensity_unfused, achieved_unfused),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text((intensity_unfused * intensity_fused)**0.5, achieved_unfused * 0.5,
        f'Fusion: {intensity_fused/intensity_unfused:.1f}x more intense',
        fontsize=10, color='purple', ha='center')

ax.set_xlabel('Arithmetic Intensity (FLOPS/byte)', fontsize=12, fontweight='bold')
ax.set_ylabel('Performance (GFLOPS)', fontsize=12, fontweight='bold')
ax.set_title('Kernel Fusion Moves Ops Rightward on the Roofline', fontsize=14, fontweight='bold')
ax.grid(True, which='both', alpha=0.3, linestyle=':')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0.1, 1000)
ax.set_ylim(10, 500000)
plt.tight_layout()
plt.show()

## Revision Notes

*Fill this in after running all sections.*

---

**Why unfused ops are slow:**



**Manual fusion speedup:**



**torch.compile speedup on transformer block:**



**CUDA graph speedup:**



**Best fusion pattern:**



**When torch.compile is NOT worth it:**



**How to fix a graph break:**



**Roofline: how fusion changes intensity:**
- Unfused intensity: ___ FLOPS/byte
- Fused intensity: ___ FLOPS/byte
- Both still ___-bound, but fused is ___x closer to ridge

**What surprised me:**



---
*Next: Chapter 7 — Weight Quantization*